# 对接 PyTorch 项目实践

前序内容介绍了 AutoFuse 的使能与配置方法，本节转向 Inductor 对接路线，以 `Add → Ge` 模型为例（输入为三个 `[128, 50]` 张量，先执行逐元素加法再执行逐元素比较，输出布尔张量），完成依赖准备、模型编译、NPU 执行、数值校验、融合结果检查、性能分析和常见问题排查，并确认模型实际进入 Inductor 编译路线，数值结果和 Profiling 产物符合预期。

本节学习大纲如下：

- 准备 Inductor NPU 扩展
- 执行 PyTorch 自动融合用例
- 验证运行与融合结果
- PyTorch 性能分析
- PyTorch 常见问题

## 1. 准备 Inductor NPU 扩展

不同环境的 `torch` 和 `torch_npu` 版本可能不一致，甚至无法正常导入。为保证用例稳定执行，本节创建课程专用虚拟环境 `./tmp/venv`，统一使用课程验证过的版本组合（`torch==2.9.0` + `torch_npu==2.9.0`），与 Notebook 内核及宿主机环境隔离。

安装单元以 `%%bash` 脚本执行：创建虚拟环境并安装依赖（`torch`、`torch_npu`、`numpy`、`pyyaml`、`setuptools`），清理旧的扩展安装和 `tmp/torchair` 目录，克隆 TorchAir 源码并回退到课程验证使用的提交，最后以 `--no-deps` 可编辑方式安装 `inductor_npu_ext`（`torch` 由虚拟环境统一管理，避免 pip 擅自改动其版本）。

In [ ]:
%%bash
set -eo pipefail

# 创建课程专用虚拟环境（与 Notebook 内核及宿主机环境隔离）
python3 -m venv ./tmp/venv

# 安装课程验证过的版本组合
./tmp/venv/bin/pip install torch==2.9.0 torch_npu==2.9.0 numpy pyyaml setuptools -i https://pypi.tuna.tsinghua.edu.cn/simple

# 清理旧的扩展安装与源码目录，保证每次安装结果一致
./tmp/venv/bin/pip uninstall -y inductor_npu_ext || true
rm -rf ./tmp/torchair

# 克隆 TorchAir 源码，并回退到课程验证使用的提交
git clone https://gitcode.com/Ascend/torchair.git ./tmp/torchair
git -C ./tmp/torchair reset --hard 82314244298ed96c086fb41742614a97ed927523

# 以可编辑方式安装 inductor_npu_ext（--no-deps：torch 已由虚拟环境统一安装）
./tmp/venv/bin/pip install --no-deps -e ./tmp/torchair/experimental/_inductor_npu_ext/python/

## 2. 执行 PyTorch 自动融合用例

下面的单元先将完整用例写入 `./tmp/autofuse_pytorch_example.py`，再用虚拟环境的 Python 在子进程中执行。子进程方式不依赖 Notebook 内核自带的 torch 环境，每次运行都是全新进程，也不受内核中残留模块状态影响。

脚本按以下顺序执行：

1. 导入 `torch`、`torch_npu` 和 `inductor_npu_ext`，其中导入 `inductor_npu_ext` 是 Inductor 路线使能 AutoFuse 的关键。
2. 构造 `Add → Ge` 模型，调用 `torch.compile` 编译，并创建 `[128, 50]` 输入。
3. 在 NPU 上以 Profiling 方式连续执行 100 步。
4. 打印输出形状、数据类型和 Profiling 目录，并与 Eager 参考结果对比。

执行单元先切换到 `tmp` 目录，再使用虚拟环境的 Python 启动用例，脚本内相对路径以 `tmp` 为基准；CANN 环境由 Notebook 运行环境自动加载，无需在单元内手动加载。

In [ ]:
%%writefile ./tmp/autofuse_pytorch_example.py
import os

import torch
import torch_npu
import torch.nn as nn
import inductor_npu_ext

_ = inductor_npu_ext  # 通过 import 副作用注册 Inductor 路线的 AutoFuse 编译扩展

DEVICE = "npu:0"
PROFILING_DIR = os.path.abspath("./profiling")


class MyModel(nn.Module):
    """Add → Ge 逐元素计算模型，输出布尔张量。"""

    def forward(self, x, y, z):
        return torch.ge(torch.add(x, y), z)


def run_model(step=100):
    """编译模型，在 NPU 上以 Profiling 方式执行，并返回输出和 Eager 对比结果。"""
    torch.npu.set_device(DEVICE)

    model = MyModel().to(DEVICE)
    # 注释下一行后重新运行执行单元，可观察 Eager 模式的性能差异
    model = torch.compile(model, dynamic=False, fullgraph=True)
    model.eval()

    x = torch.randn(128, 50, device=DEVICE)
    y = torch.randn(128, 50, device=DEVICE)
    z = torch.randn(128, 50, device=DEVICE)

    experimental_config = torch_npu.profiler._ExperimentalConfig(
        export_type=[torch_npu.profiler.ExportType.Text],
        profiler_level=torch_npu.profiler.ProfilerLevel.Level2,
        msprof_tx=False,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
        l2_cache=False,
        op_attr=False,
        data_simplification=False,
        record_op_args=False,
        gc_detect_threshold=None,
    )

    with torch_npu.profiler.profile(
        activities=[torch_npu.profiler.ProfilerActivity.CPU, torch_npu.profiler.ProfilerActivity.NPU],
        on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(PROFILING_DIR),
        record_shapes=True,
        profile_memory=False,
        with_stack=False,
        with_modules=False,
        with_flops=False,
        experimental_config=experimental_config,
    ) as prof:
        for _ in range(step):
            result = model(x, y, z)

    # 使用相同输入计算 Eager 参考结果，确认融合路径的数值正确性
    with torch.no_grad():
        expected = MyModel().to(DEVICE)(x, y, z)

    torch.npu.synchronize()
    equals_eager = torch.equal(result, expected)
    return result, equals_eager


result, equals_eager = run_model()
print("PyTorch version:", torch.__version__)
print("torch_npu version:", torch_npu.__version__)
print("Output shape:", tuple(result.shape))
print("Output dtype:", result.dtype)
print("Profiling directory:", PROFILING_DIR)
print("Output equals Eager reference:", equals_eager)

In [ ]:
%%bash
set -eo pipefail

# 切换到 tmp 目录执行，使脚本内相对路径以 tmp 为基准
cd ./tmp

# CANN 环境已由 Notebook 运行环境自动加载，直接使用课程虚拟环境的 Python 执行用例
./venv/bin/python ./autofuse_pytorch_example.py

## 3. 验证运行与融合结果

代码执行完成后，需要分别验证“计算是否正确”和“算子是否融合”。二者关注点不同，不能只以代码无报错作为融合成功的依据。

### 3.1 理解运行输出

脚本执行完成后会打印以下信息：

<table align="left">
<thead><tr><th>输出项</th><th>预期观察</th><th>说明</th></tr></thead>
<tbody>
<tr><td><code>PyTorch version</code>、<code>torch_npu version</code></td><td><code>2.9.0+cpu</code>、<code>2.9.0</code></td><td>版本由安装单元固定，用于确认执行环境是否符合本例要求</td></tr>
<tr><td><code>Output shape</code></td><td><code>(128, 50)</code></td><td>输出形状应与输入形状一致</td></tr>
<tr><td><code>Output dtype</code></td><td><code>torch.bool</code></td><td><code>Ge</code> 为逐元素比较算子，输出为布尔张量</td></tr>
<tr><td><code>Profiling directory</code></td><td><code>tmp</code> 下的 <code>profiling</code> 目录路径</td><td>后续从该目录检查算子执行与融合信息</td></tr>
<tr><td><code>Output equals Eager reference</code></td><td><code>True</code></td><td>表示编译模型输出与 Eager 参考结果一致；为 <code>False</code> 时说明数值校验未通过</td></tr>
</tbody>
</table>
<div style="clear:left"></div>

### 3.2 验证数值结果

脚本使用 `torch.equal` 对比编译模型输出和相同输入下的 Eager 参考结果。结果为 `True` 说明编译路径与 Eager 路径的输出完全一致；为 `False` 时，应先排查输入、数据类型、模型状态和首个产生差异的算子，再分析融合行为。

### 3.3 验证融合结果

运行完成后，`tmp` 目录下会生成 `profiling` 目录及相关 Profiling 文件。PyTorch Profiling 导出结果通常位于：

```text
./tmp/profiling/
└── 主机标识_进程号_时间戳_ascend_pt/
    └── PROF_000001_时间戳_随机串/
        └── mindstudio_profiler_output/
            └── op_summary_*.csv
```

打开本次运行对应的 `op_summary_*.csv`，即可查看本次执行的算子详情。如果算子列表中出现名称类似 `autofused_add_ge_拓扑哈希` 的 Kernel，说明 `Add` 和 `Ge` 已成功融合为一个融合算子；如果仍能看到 `Add`、`Ge` 两个独立算子，则说明本次运行未实现预期融合。

### 3.4 算子自动融合执行结果

<div style="text-align:left">
<img src="./images/pytorch_autofuse.png" alt="Inductor 路线自动融合执行结果" width="80%">
</div>

该图展示开启 `torch.compile` 并使能 AutoFuse 后的 Profiling 结果。`Add` 和 `Ge` 以一个融合 Kernel 的形式执行，说明自动融合已经生效。

### 3.5 单算子执行结果

想获取未融合的对照结果，可在 `./tmp/autofuse_pytorch_example.py` 中注释掉 `model = torch.compile(model, dynamic=False, fullgraph=True)` 一行，再重新运行执行单元。
<div style="text-align:left">
<img src="./images/pytorch_no_autofuse.png" alt="Inductor 路线单算子执行结果" width="60%">
</div>

该图展示未调用 `torch.compile` 时的 Profiling 结果。`Add` 和 `Ge` 分别以独立算子的形式执行，可作为融合场景的对照结果。


对比两张图片可以看到：在融合场景中，`Add` 和 `Ge` 由一个融合 Kernel 执行；在单算子场景中，两个算子分别执行。评估融合收益时，应比较融合 Kernel 的平均任务执行时间与单算子场景下 `Add`、`Ge` 平均任务执行时间之和。如果前者小于后者，说明本次实验中融合减少了整体执行开销并取得了性能收益。为保证结果可比，应使用相同的输入、迭代次数和 Profiling 配置，并在完成预热后统计稳态执行时间，不计入首次图编译开销。

## 4. PyTorch 性能分析

本例的性能分析需要区分首次编译开销和稳态执行耗时。首次执行 `torch.compile` 编译的模型时，会触发图捕获、编译和 Kernel 生成，耗时明显偏长，不应直接作为稳态性能结果。

建议按以下方式进行对比：

1. 准备融合与基线两种场景：融合场景调用 `torch.compile` 使能 AutoFuse；基线场景不调用 `torch.compile`，以 Eager 方式直接执行（可参考 3.5 节，注释掉 `torch.compile` 一行后重新运行）。
2. 两种场景使用相同的输入、数据类型、执行步数、预热方式和 Profiling 配置，保证结果可比。
3. 完成预热后再采样：排除首次编译开销，比较多轮迭代的平均执行时间。
4. 结合 Profiling 中的 Kernel 数量、融合 Kernel 名称和平均任务执行时间综合判断。

出现 `autofused_` 开头的融合 Kernel 只能说明算子发生了融合。只有当融合 Kernel 的稳态执行时间低于基线场景中 `Add` 和 `Ge` 等独立算子的执行时间之和时，才能说明本次实验取得了性能收益。

## 5. PyTorch 常见问题

- **扩展无法导入**：确认安装单元已成功执行，且执行单元在 `tmp` 目录下使用 `./venv/bin/python` 启动用例；必要时重新运行安装单元。
- **编译失败或未发生融合**：确认 `inductor_npu_ext` 已在编译前导入，检查 `torch.compile` 配置、输入形状及目标算子是否在支持范围内，并结合编译日志和 Profiling 判断。
- **性能无收益或波动较大**：排除首次编译开销，完成预热后多次采样，并保持输入、执行步数、Profiling 配置、软件版本和设备状态一致。
- **精度异常**：使用相同输入对比 Eager 和编译模型的执行结果，确认模型状态、数据类型和参考计算一致，再定位首个产生差异的算子。

## 课后练习

请根据本节实践内容完成以下题目进行自测。

1. （判断题）PyTorch 用例执行无报错即可证明 `Add` 和 `Ge` 已完成自动融合，无需查看 Profiling 数据。

2. （多选题）本例进入 Inductor Ascend NPU 编译路径的关键步骤有哪些？
    A. 导入 `inductor_npu_ext`
    B. 调用 `torch.compile`
    C. 设置 `AUTOFUSE_FLAGS` 环境变量
    D. 删除 `sys.modules` 中的 `torch`

3. （多选题）验证本例执行结果时应关注哪些内容？
    A. 编译模型输出是否与 Eager 参考结果一致
    B. 输出形状和数据类型是否符合预期
    C. Profiling 中是否存在预期的 NPU 执行或融合记录
    D. 只要代码无报错即可确认融合成功

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.03_answer.txt